# Прогноз потребления электроэнергии

Ноутбук подготовлен для Google Colab и GitHub. Он показывает все результаты внутри ячеек и не создает папки с артефактами.


## Запуск в Google Colab

Сначала выполните ячейку ниже, затем запускайте ноутбук сверху вниз.


In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip -q install pandas numpy matplotlib scikit-learn statsmodels

print("Среда готова.")


## Импорты и загрузка данных


In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.ensemble import IsolationForest
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from statsmodels.tsa.api import ExponentialSmoothing
from statsmodels.tsa.seasonal import STL

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 30)

LOCAL_PREPARED = Path("data/electricity_daily_prepared.csv")
RAW_URL = "https://raw.githubusercontent.com/MVRonkin/TimeSeriesCourse/main/OLD%20Versions/2026/datasets/Electricity%20Consumption%20and%20Production%20Data/Electricity.csv"
RANDOM_STATE = 42


def prepare_daily(raw):
    df = raw.copy()
    df["DateTime"] = pd.to_datetime(df["DateTime"])
    numeric_cols = [col for col in df.columns if col != "DateTime"]
    df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors="coerce")
    df = df.sort_values("DateTime").drop_duplicates("DateTime")
    hourly = df.set_index("DateTime").asfreq("h")
    hourly[numeric_cols] = hourly[numeric_cols].interpolate(limit_direction="both")
    daily = hourly.resample("D").mean().reset_index()
    daily = daily.rename(columns={"DateTime": "ds", "Consumption": "y"})
    return daily[["ds", "y"]].dropna().reset_index(drop=True)


def load_electricity():
    if LOCAL_PREPARED.exists():
        df = pd.read_csv(LOCAL_PREPARED)
        source = str(LOCAL_PREPARED)
    else:
        raw = pd.read_csv(RAW_URL)
        df = prepare_daily(raw)
        source = RAW_URL
    df["ds"] = pd.to_datetime(df["ds"])
    df["y"] = pd.to_numeric(df["y"], errors="coerce")
    return df.dropna().sort_values("ds").reset_index(drop=True), source


daily_df, source = load_electricity()
print(f"Источник данных: {source}")
print(f"Размер ряда: {daily_df.shape[0]} дней")
display(daily_df.head())


## Проверка качества и EDA


In [ ]:
quality = pd.DataFrame({
    "metric": ["start", "end", "rows", "missing_y", "duplicate_dates", "frequency_mode"],
    "value": [
        daily_df["ds"].min(),
        daily_df["ds"].max(),
        len(daily_df),
        int(daily_df["y"].isna().sum()),
        int(daily_df["ds"].duplicated().sum()),
        daily_df["ds"].diff().mode().iloc[0],
    ],
})
display(quality)
display(daily_df["y"].describe().to_frame("daily_consumption"))

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(daily_df["ds"], daily_df["y"], linewidth=1)
ax.set_title("Среднесуточное потребление электроэнергии")
ax.set_xlabel("Дата")
ax.set_ylabel("Consumption")
ax.grid(alpha=0.25)
plt.show()


In [ ]:
eda = daily_df.copy()
eda["year"] = eda["ds"].dt.year
eda["month"] = eda["ds"].dt.month
eda["dayofweek"] = eda["ds"].dt.dayofweek

monthly = eda.groupby("month")["y"].agg(["count", "mean", "std", "min", "max"]).round(3)
weekly = eda.groupby("dayofweek")["y"].agg(["count", "mean", "std", "min", "max"]).round(3)
yearly = eda.groupby("year")["y"].agg(["count", "mean", "std", "min", "max"]).round(3)

print("Месячная сезонность")
display(monthly)
print("Недельная сезонность")
display(weekly)
print("Годовая статистика")
display(yearly)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
monthly["mean"].plot(ax=axes[0], marker="o", title="Среднее по месяцам")
weekly["mean"].plot(ax=axes[1], marker="o", title="Среднее по дням недели")
for ax in axes:
    ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()


## Сравнение моделей прогноза


In [ ]:
HORIZON = 7
train = daily_df.iloc[:-HORIZON].copy()
test = daily_df.iloc[-HORIZON:].copy()


def make_features(data):
    out = data.copy()
    for lag in [1, 2, 3, 7, 14, 28]:
        out[f"lag_{lag}"] = out["y"].shift(lag)
    for window in [7, 14, 28]:
        out[f"roll_mean_{window}"] = out["y"].shift(1).rolling(window).mean()
        out[f"roll_std_{window}"] = out["y"].shift(1).rolling(window).std()
    out["dayofweek"] = out["ds"].dt.dayofweek
    out["month"] = out["ds"].dt.month
    return out.dropna().reset_index(drop=True)


FEATURES = [
    "lag_1", "lag_2", "lag_3", "lag_7", "lag_14", "lag_28",
    "roll_mean_7", "roll_std_7", "roll_mean_14", "roll_std_14",
    "roll_mean_28", "roll_std_28", "dayofweek", "month",
]


def metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "MAPE_%": np.mean(np.abs((y_true - y_pred) / np.maximum(np.abs(y_true), 1e-8))) * 100,
    }


def recursive_forecast(model, history, horizon, scaler=None):
    hist = history.copy().reset_index(drop=True)
    preds = []
    for _ in range(horizon):
        row = pd.DataFrame({"ds": [hist["ds"].iloc[-1] + pd.Timedelta(days=1)], "y": [np.nan]})
        candidate = pd.concat([hist, row], ignore_index=True)
        feats = make_features(candidate).tail(1)
        x = feats[FEATURES].to_numpy()
        if scaler is not None:
            x = scaler.transform(x)
        pred = float(model.predict(x)[0])
        preds.append(pred)
        hist.loc[len(hist)] = {"ds": row["ds"].iloc[0], "y": pred}
    return np.array(preds)


feature_train = make_features(train)
X_train = feature_train[FEATURES].to_numpy()
y_train = feature_train["y"].to_numpy()
y_test = test["y"].to_numpy()

results = []
forecast_table = test[["ds", "y"]].rename(columns={"y": "actual"}).reset_index(drop=True)

baselines = {
    "Naive": np.repeat(train["y"].iloc[-1], HORIZON),
    "SeasonalNaive_7": train["y"].iloc[-7:].to_numpy(),
    "MovingAverage_14": np.repeat(train["y"].tail(14).mean(), HORIZON),
}
for name, pred in baselines.items():
    results.append({"group": "Baseline", "model": name, **metrics(y_test, pred)})
    forecast_table[name] = pred

try:
    exp_model = ExponentialSmoothing(train["y"], trend="add", seasonal="add", seasonal_periods=7).fit()
    exp_pred = exp_model.forecast(HORIZON).to_numpy()
    results.append({"group": "Statistical", "model": "ExpSmoothing", **metrics(y_test, exp_pred)})
    forecast_table["ExpSmoothing"] = exp_pred
except Exception as exc:
    print(f"ExpSmoothing пропущен: {exc}")

for name, model, group, use_scaler in [
    ("LinearRegression", LinearRegression(), "ML", False),
    ("Ridge_alpha_25", Ridge(alpha=25.0), "ML", False),
    ("KNN_15", KNeighborsRegressor(n_neighbors=15), "ML", True),
    ("MLP_small", MLPRegressor(hidden_layer_sizes=(16,), max_iter=400, random_state=RANDOM_STATE), "DL", True),
    ("MLP_medium", MLPRegressor(hidden_layer_sizes=(32, 16), max_iter=400, random_state=RANDOM_STATE), "DL", True),
]:
    scaler = StandardScaler() if use_scaler else None
    X_fit = scaler.fit_transform(X_train) if scaler else X_train
    model.fit(X_fit, y_train)
    pred = recursive_forecast(model, train, HORIZON, scaler=scaler)
    results.append({"group": group, "model": name, **metrics(y_test, pred)})
    forecast_table[name] = pred

results_df = pd.DataFrame(results).sort_values("MAE").reset_index(drop=True)
display(results_df.round(4))
display(forecast_table.round(4))

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(train["ds"].tail(60), train["y"].tail(60), label="train tail")
ax.plot(test["ds"], y_test, marker="o", label="actual")
for col in results_df["model"].head(3):
    ax.plot(forecast_table["ds"], forecast_table[col], marker="o", label=col)
ax.set_title("Сравнение лучших прогнозов")
ax.grid(alpha=0.25)
ax.legend()
plt.show()


## Анализ аномалий


In [ ]:
anom = daily_df.copy()
stl = STL(anom["y"], period=7, robust=True).fit()
anom["residual"] = stl.resid
q1, q3 = anom["residual"].quantile([0.25, 0.75])
iqr = q3 - q1
anom["Seasonal_IQR"] = (anom["residual"] < q1 - 1.5 * iqr) | (anom["residual"] > q3 + 1.5 * iqr)

rolling_mean = anom["y"].rolling(30, min_periods=14).mean()
rolling_std = anom["y"].rolling(30, min_periods=14).std()
z = (anom["y"] - rolling_mean) / rolling_std
anom["Rolling_Z"] = z.abs() > 3

iso_features = make_features(anom)[FEATURES]
iso = IsolationForest(contamination=0.02, random_state=RANDOM_STATE)
iso_pred = iso.fit_predict(iso_features)
anom["IsolationForest"] = False
anom.loc[iso_features.index, "IsolationForest"] = iso_pred == -1

summary = pd.DataFrame({
    "method": ["Seasonal_IQR", "Rolling_Z", "IsolationForest"],
    "anomalies": [int(anom["Seasonal_IQR"].sum()), int(anom["Rolling_Z"].sum()), int(anom["IsolationForest"].sum())],
})
display(summary)

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(anom["ds"], anom["y"], linewidth=1, label="y")
points = anom[anom["Seasonal_IQR"]]
ax.scatter(points["ds"], points["y"], color="crimson", s=18, label="Seasonal_IQR")
ax.set_title("Найденные аномалии")
ax.grid(alpha=0.25)
ax.legend()
plt.show()


## Итог

Ноутбук показывает полный цикл анализа: подготовка дневного ряда, EDA, сравнение моделей прогноза и поиск аномалий. Все результаты остаются внутри ноутбука.
